> **2001–2019 version for the 9 Nov meeting (made 2026-11-05, moved 2026-11-06).** Same notebook as `analysis/radiative_forcing.ipynb`, but years **2001–2019** (19 paired years; 2015–2019 = extension nudged with the MET 2000–2025 data). Self-contained: input `analysis-2001-2019/processed/`, figures and tables in `analysis-2001-2019/results/`; nothing outside this folder is read or written except the repo code and `diagnostics/lcc-modified-cells/modified_cells.nc`. Built by `prepare.sh` (see `README.md`). Adds the **design A** FUT row `FUT_fBVOClccFUT` (LCC_FUT − LCC_FUT_fBVOClccFUT) only if that run's 2001–2019 file exists (not expected before the meeting).

# Radiative forcing: Ghan decomposition

Polar maps north of 45°N. Plotting code: `boreal_forest_expansion.plotting.forcing_maps`.

- **Total** = LCC − CTRL of the same period (FUT uses CTRL_FUT). **BVOC** = LCC − LCC_fBVOC.
- **Ghan (2013)** split of the top-of-model flux difference (positive = down, warming):
  d(total) = d(ARI: F − F_clean) + d(cloud: F_clean − F_clean,clear) + d(clear-clean: F_clean,clear).
  The clear-clean term holds surface albedo **and** every other clear-sky, aerosol-free change (gases, temperature, water vapour; LW).
  These runs are nudged with interactive land, so the differences are effective (fast-response-inclusive) effects, not instantaneous forcings.

**Data and caveats (read before interpreting)**
- Input: monthly postprocessed files `processed/[alias]_[atm|lnd]_[CATEGORY]_2001-2019.nc` (first 12 months, year 2000, dropped as spin-up), made from the work archive h0 files 2000-01 … 2019-12.
  Runs = the CO₂-fix chain, the same 11 cases as the 2001–2014 set: CTRL_PD `…nudg_ctrl_…-20261029`, LCC_PD `…nudg_lcc_…-20261028`, CTRL_FUT / LCC_FUT `-20261028`, fBVOC runs `-20261031` (all 19 MEGAN species prescribed; list in `cases.txt`).
- fBVOC runs prescribe a **2001–2014 climatology** of the source run's emissions; in 2015–2019 the same climatology is repeated, while the interactive runs see 2015–2019 weather. (b, inferred from the setup, not checked in the files)
- CTRL and LCC have different land spin-ups (LCC: extra 100-yr land-only BGC spin-up) → LCC − CTRL contains a spin-up offset (large in FUT).
- `LCC_FUT − CTRL_PD` = LCC **plus** the whole 2100 background change (SST, GHG, CO₂, emissions): dominated by climate change.
- `LCC_FUT − LCC_FUT_fBVOC`: fBVOC_FUT uses CTRL_PD emissions → mixes the climate-driven emission change with the LCC effect; **not** a forest-expansion-only BVOC effect.
- fBVOC runs have MEGAN off: all 19 MEGAN species come from a climatology of the source run (CTRL_PD, CTRL_FUT or LCC_PD) (no `MEG_*` output, no interannual or weather coupling of the emissions).
- Maps: mean of the 19 paired yearly differences. **Grey `/////` hatching = NOT significant** (paired t-test, Benjamini–Hochberg FDR α = 0.10 over the drawn cells ≥45°N).
  Text under each panel: area-weighted mean ≥45°N ± 95 % t-interval across years (land-only fields: land mean).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # repo src/, no pip install needed

import matplotlib.pyplot as plt
from boreal_forest_expansion.plotting.style import set_default_style
from boreal_forest_expansion.plotting import forcing_maps as fm

set_default_style()
START_YEAR, END_YEAR = 2001, 2019
# 2001-2019 set (meeting 9 Nov 2026): everything inside analysis-2001-2019/ (delete the folder to remove it all)
HERE = Path.cwd()                                   # analysis-2001-2019/
fm.POSTPROCESSED_DIR = HERE / "processed"
fm.FIGURE_DIR = HERE / "results" / "figures"
fm.FIG_LCC_CTRL, fm.FIG_BACKGROUND = fm.FIGURE_DIR / "LCC-CTRL", fm.FIGURE_DIR / "CTRL_FUT-CTRL_PD"
fm.FIG_FBVOC = fm.FIGURE_DIR / "fBVOC"
fm.FIG_FBVOC_PRE, fm.FIG_FBVOC_B = fm.FIG_FBVOC / "pre-artefact-correction", fm.FIG_FBVOC / "artefact-design-B"
fm.TABLE_DIR = HERE / "results" / "tables"
fm.TABLE_FBVOC_PRE, fm.TABLE_FBVOC_B = fm.TABLE_DIR / "pre-artefact-correction", fm.TABLE_DIR / "artefact-design-B"
for _d in (fm.FIG_LCC_CTRL, fm.FIG_BACKGROUND, fm.FIG_FBVOC_PRE, fm.FIG_FBVOC_B, fm.TABLE_FBVOC_PRE, fm.TABLE_FBVOC_B):
    _d.mkdir(parents=True, exist_ok=True)
common = dict(start_year=START_YEAR, end_year=END_YEAR, extent_lat=45, mean_lat_min=45, save=True)
# uncorrected fBVOC figures (no artefact correction) live in pre-artefact-correction/ (user 2026-11-03);
# LCC − CTRL -> results/figures/LCC-CTRL/, CTRL_FUT − CTRL_PD -> results/figures/CTRL_FUT-CTRL_PD/ (here: analysis-2001-2019/results/) (figdir=None routes by name)
PRE_DIR = fm.FIG_FBVOC_PRE
common["figdir"] = PRE_DIR
print("Figures ->", fm.FIGURE_DIR)
print(fm.BVOC_PAIRS)

for exp, roles in fm.BVOC_PAIRS.items():
    for alias in roles.values():
        path = fm.filename(alias, "RADIATIVE", START_YEAR, END_YEAR)
        print(path.name, "✓" if path.exists() else "MISSING")

# forcing summary figures, named after the fBVOC run of their BVOC panels
# FUT: all three FUT fBVOC runs while the main FUT emission source is on hold (user 2026-11-04)
SUMMARY = {"PD_fBVOCctrlPD": fm.BVOC_PAIRS["PD"],
           "FUT_fBVOCctrlPD": fm.BVOC_PAIRS["FUT"],
           "FUT_fBVOCctrlFUT": {**fm.BVOC_PAIRS["FUT"], "fbvoc": "LCC_FUT_fBVOCctrlFUT"},
           "FUT_fBVOClccPD": {**fm.BVOC_PAIRS["FUT"], "fbvoc": "LCC_FUT_fBVOClccPD"}}
# design A (user OK 2026-11-05): LCC_FUT's own emissions prescribed; only when its file exists
if fm.filename("LCC_FUT_fBVOClccFUT", "RADIATIVE", START_YEAR, END_YEAR).exists():
    SUMMARY["FUT_fBVOClccFUT"] = {**fm.BVOC_PAIRS["FUT"], "fbvoc": "LCC_FUT_fBVOClccFUT"}
else:
    print("LCC_FUT_fBVOClccFUT 2001-2019 not there yet: design A row skipped")
for pairs in SUMMARY.values():
    path = fm.filename(pairs["fbvoc"], "RADIATIVE", START_YEAR, END_YEAR)
    print(path.name, "✓" if path.exists() else "MISSING")

## 1. Six-panel summary (albedo proxy, Total, BVOC terms)
Moved unchanged from the old `forcing_maps.ipynb`. Panel 1 = clear-sky surface albedo forcing proxy (LCC − CTRL); panels 3–6 = LCC − LCC_fBVOC.

In [ ]:
for exp, pairs in SUMMARY.items():
    for band in ("NET", "SW", "LW"):
        fig, axes = fm.make_forcing_summary_figure(exp, band=band, pairs=pairs, transparent_background=True, **common)
        plt.show(); plt.close(fig)

### 1b. Six-panel summary with shared colour scales
Same panels and data as section 1. **`shared6`**: all six panels on one scale [−6, 6] W m⁻². **`split6-2`**: albedo + total on [−6, 6], the four BVOC-related panels (LCC − LCC_fBVOC) on [−2, 2]. File names end in `_shared6` / `_split6-2`.

In [ ]:
for scales in ("shared6", "split6-2"):   # fm.SUMMARY_SCALES
    for exp, pairs in SUMMARY.items():
        for band in ("NET", "SW", "LW"):
            fig, axes = fm.make_forcing_summary_figure(exp, band=band, pairs=pairs, scales=scales,
                                                       transparent_background=True, **common)
            plt.show(); plt.close(fig)

## 2. Ghan decomposition per contrast (total + 3 terms, shared colour scale)
One figure per contrast × band: LCC − CTRL and LCC − LCC_fBVOC for PD and FUT, plus CTRL_FUT − CTRL_PD (2100 background change alone, FUT only): 15 figures. All four panels share one colour scale, so the terms can be compared by colour and visibly add up to the total.
The closure line checks max |d(ARI) + d(cloud) + d(clear-clean) − d(total)| over all cells and years (exact by construction; ~1e-5 W m⁻² = float32 rounding).

In [ ]:
for exp in ("PD", "FUT"):
    for contrast in fm.GHAN_CONTRASTS:   # LCC-CTRL, LCC-fBVOC, CTRL_FUT-CTRL_PD (FUT only)
        if contrast == "background" and exp == "PD":
            continue
        for band in ("NET", "SW", "LW"):
            fig, axes = fm.make_ghan_figure(exp, contrast, band, figdir=PRE_DIR if contrast == "bvoc" else None, start_year=START_YEAR, end_year=END_YEAR)
            plt.show(); plt.close(fig)

## 3. Seasonal Ghan decomposition (DJF / MAM / JJA / SON)
Same four terms, one row per season, one colour scale per row. Separates the snow-masking albedo signal (spring, clear-clean SW) from the cloud and aerosol terms (summer).
Seasonal means are month-length weighted within each calendar year; **DJF = Jan + Feb + Dec of the same year**, so all 19 years stay paired.

In [ ]:
for exp in ("PD", "FUT"):
    for contrast in fm.GHAN_CONTRASTS:   # LCC-CTRL, LCC-fBVOC, CTRL_FUT-CTRL_PD (FUT only)
        if contrast == "background" and exp == "PD":
            continue
        for band in ("NET", "SW", "LW"):
            fig, axes = fm.make_ghan_figure(exp, contrast, band, figdir=PRE_DIR if contrast == "bvoc" else None, seasonal=True,
                                            start_year=START_YEAR, end_year=END_YEAR)
            plt.show(); plt.close(fig)

## 4. Single map (quick look)

In [ ]:
da = fm.difference_map("LCC_PD", fm.BVOC_PAIRS["PD"]["fbvoc"], "RADIATIVE", "NET_toa", START_YEAR, END_YEAR)
da.plot()